# Phase 4.3 — Route-Month Data Foundation

## Phase 4.3.1 — Raw T-100 Grain & Route-Month Multiplicity Investigation

Purpose: investigate why multiple legitimate T-100 Segment records can exist
for the same WN directional route-month before defining aggregation rules.

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
raw_path = Path("../data/raw/T_T100_SEGMENT_ALL_CARRIER_2024.zip")
df_2024 = pd.read_csv(raw_path)
raw_path = Path("../data/raw/T_T100_SEGMENT_ALL_CARRIER_2025.zip")
df_2025 = pd.read_csv(raw_path)
raw_path = Path("../data/raw/T_T100_SEGMENT_ALL_CARRIER_2026.zip")
df_2026 = pd.read_csv(raw_path)

In [3]:
df_historical = pd.concat([df_2024, df_2025, df_2026], ignore_index=True)
print("Shape of df_historical:", df_historical.shape)
df_historical.head()

Shape of df_historical: (1356516, 50)


,DEPARTURES_SCHEDULED,DEPARTURES_PERFORMED,PAYLOAD,SEATS,PASSENGERS,FREIGHT,MAIL,DISTANCE,RAMP_TO_RAMP,AIR_TIME,...,DEST_WAC,AIRCRAFT_GROUP,AIRCRAFT_TYPE,AIRCRAFT_CONFIG,YEAR,QUARTER,MONTH,DISTANCE_GROUP,CLASS,DATA_SOURCE
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,15.0,0.0,0.0,...,41,6,622,1,2024,4,10,1,L,DU
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.0,0.0,0.0,...,15,0,79,1,2024,1,1,1,F,DU
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.0,0.0,0.0,...,15,0,79,1,2024,3,7,1,F,DU
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.0,0.0,0.0,...,15,0,79,1,2024,3,8,1,F,DU
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.0,0.0,0.0,...,15,0,79,1,2024,4,12,1,F,DU


In [4]:
del df_2024, df_2025, df_2026

In [5]:
wn_f_historical = df_historical[
    (df_historical['UNIQUE_CARRIER'] == 'WN') &
     (df_historical['CLASS'] == 'F') 
        ].copy()

print("shape of wn_f_historical:", wn_f_historical.shape)
wn_f_historical.head()

shape of wn_f_historical: (139082, 50)


,DEPARTURES_SCHEDULED,DEPARTURES_PERFORMED,PAYLOAD,SEATS,PASSENGERS,FREIGHT,MAIL,DISTANCE,RAMP_TO_RAMP,AIR_TIME,...,DEST_WAC,AIRCRAFT_GROUP,AIRCRAFT_TYPE,AIRCRAFT_CONFIG,YEAR,QUARTER,MONTH,DISTANCE_GROUP,CLASS,DATA_SOURCE
37979,0.0,1.0,34600.0,143.0,0.0,0.0,0.0,29.0,29.0,9.0,...,91,6,612,1,2024,4,10,1,F,DU
37980,0.0,1.0,34600.0,143.0,0.0,0.0,0.0,104.0,66.0,24.0,...,33,6,612,1,2024,1,1,1,F,DU
37981,0.0,1.0,34600.0,143.0,0.0,0.0,0.0,109.0,93.0,34.0,...,91,6,612,1,2024,1,2,1,F,DU
37982,0.0,1.0,34600.0,143.0,0.0,0.0,0.0,229.0,49.0,39.0,...,42,6,612,1,2024,3,8,1,F,DU
37983,0.0,1.0,34600.0,143.0,0.0,0.0,0.0,241.0,51.0,42.0,...,22,6,612,1,2024,1,3,1,F,DU


In [6]:
wn_f_historical[['YEAR', 'MONTH']].drop_duplicates().sort_values(['YEAR', 'MONTH']).reset_index(drop=True)

,YEAR,MONTH
0,2024,1
1,2024,2
2,2024,3
3,2024,4
4,2024,5
5,2024,6
6,2024,7
7,2024,8
8,2024,9
9,2024,10


In [7]:
route_month_row_counts  = (
    wn_f_historical
    .groupby(["YEAR", "MONTH", "ORIGIN", "DEST"])
    .size()
    .reset_index(name = 'RAW_ROW_COUNT')
    )

print(route_month_row_counts.head(10))
print(route_month_row_counts.shape)

   YEAR  MONTH ORIGIN DEST  RAW_ROW_COUNT
0  2024      1    ABQ  AUS              3
1  2024      1    ABQ  BUR              1
2  2024      1    ABQ  BWI              3
3  2024      1    ABQ  DAL              3
4  2024      1    ABQ  DEN              3
5  2024      1    ABQ  HOU              3
6  2024      1    ABQ  LAS              3
7  2024      1    ABQ  LAX              3
8  2024      1    ABQ  LGB              3
9  2024      1    ABQ  MCI              1
(55395, 5)


In [8]:
route_month_row_counts["RAW_ROW_COUNT"].value_counts().sort_index()

RAW_ROW_COUNT
1    10419
2     6464
3    38342
4      144
5       23
6        3
Name: count, dtype: int64

In [9]:
print('single_row_route_months: ',len(route_month_row_counts[route_month_row_counts["RAW_ROW_COUNT"] == 1]))
multi_row_route_months = route_month_row_counts[route_month_row_counts["RAW_ROW_COUNT"] > 1]
print('multi_row_route_months: ',len(multi_row_route_months))
print('multi_row_route_months percentage: ',len(multi_row_route_months)/len(route_month_row_counts)*100)


single_row_route_months:  10419
multi_row_route_months:  44976
multi_row_route_months percentage:  81.19144327105334


In [10]:
route_month_row_counts[
    route_month_row_counts["RAW_ROW_COUNT"] == 2
].head()


,YEAR,MONTH,ORIGIN,DEST,RAW_ROW_COUNT
10,2024,1,ABQ,MCO,2
24,2024,1,AMA,AUS,2
44,2024,1,ATL,IAD,2
83,2024,1,AUS,CUN,2
121,2024,1,BDL,DCA,2


In [11]:
route_month_row_counts[
    route_month_row_counts["RAW_ROW_COUNT"] == 6
].head()

,YEAR,MONTH,ORIGIN,DEST,RAW_ROW_COUNT
31815,2025,5,SLC,DEN,6
37035,2025,8,FLL,MCO,6
50609,2026,3,MCO,BWI,6


In [12]:
abq_mco_2024_01 = wn_f_historical.loc[
    (wn_f_historical["YEAR"] == 2024) &
    (wn_f_historical["MONTH"] == 1) &
    (wn_f_historical["ORIGIN"] == "ABQ") &
    (wn_f_historical["DEST"] == "MCO"),
    [
        "UNIQUE_CARRIER",
        "YEAR",
        "MONTH",
        "ORIGIN",
        "DEST",
        "AIRCRAFT_GROUP",
        "AIRCRAFT_TYPE",
        "AIRCRAFT_CONFIG",
        "UNIQUE_CARRIER_ENTITY",
        "DATA_SOURCE",
        "DISTANCE",
        "PASSENGERS",
        "SEATS",
        "DEPARTURES_SCHEDULED",
        "DEPARTURES_PERFORMED"
    ]
].sort_values("AIRCRAFT_TYPE").reset_index(drop=True)

abq_mco_2024_01

,UNIQUE_CARRIER,YEAR,MONTH,ORIGIN,DEST,AIRCRAFT_GROUP,AIRCRAFT_TYPE,AIRCRAFT_CONFIG,UNIQUE_CARRIER_ENTITY,DATA_SOURCE,DISTANCE,PASSENGERS,SEATS,DEPARTURES_SCHEDULED,DEPARTURES_PERFORMED
0,WN,2024,1,ABQ,MCO,6,612,1,06725,DU,1553.0,402.0,429.0,3.0,3.0
1,WN,2024,1,ABQ,MCO,6,614,1,06725,DU,1553.0,145.0,175.0,1.0,1.0


In [13]:
aircraft_type_nunique = (
    wn_f_historical
    .groupby(["YEAR", "MONTH", "ORIGIN", "DEST"])["AIRCRAFT_TYPE"]
    .nunique()
    .reset_index(name="AIRCRAFT_TYPE_NUNIQUE")
)

In [14]:
aircraft_type_nunique

,YEAR,MONTH,ORIGIN,DEST,AIRCRAFT_TYPE_NUNIQUE
0,2024,1,ABQ,AUS,3
1,2024,1,ABQ,BUR,1
2,2024,1,ABQ,BWI,3
3,2024,1,ABQ,DAL,3
4,2024,1,ABQ,DEN,3
...,...,...,...,...,...
55390,2026,5,VPS,DAL,3
55391,2026,5,VPS,HOU,3
55392,2026,5,VPS,JAN,1
55393,2026,5,VPS,MCI,2


In [15]:
route_month_row_counts = route_month_row_counts.merge(
    aircraft_type_nunique,
    how="inner",
    on=["YEAR", "MONTH", "ORIGIN", "DEST"]
)



In [16]:
single_type_multi_row = route_month_row_counts[
    (route_month_row_counts["RAW_ROW_COUNT"] > 1) &
    (route_month_row_counts["AIRCRAFT_TYPE_NUNIQUE"] == 1)
]
print(len(single_type_multi_row))
single_type_multi_row


2


,YEAR,MONTH,ORIGIN,DEST,RAW_ROW_COUNT,AIRCRAFT_TYPE_NUNIQUE
28134,2025,3,TPA,MCO,2,1
33217,2025,6,MCO,TPA,2,1


In [17]:
tpa_mco_2025_03 = wn_f_historical.loc[
    (wn_f_historical["ORIGIN"] == "TPA") &
    (wn_f_historical["DEST"] == "MCO") &
    (wn_f_historical["MONTH"] == 3) &
    (wn_f_historical["YEAR"] == 2025),
    [
    "UNIQUE_CARRIER",
    "UNIQUE_CARRIER_NAME",
    "CARRIER",
    "CARRIER_NAME",
    "UNIQUE_CARRIER_ENTITY",
    "REGION",
    "AIRCRAFT_TYPE",
    "AIRCRAFT_CONFIG",
    "DATA_SOURCE",
    "DISTANCE",
    "PASSENGERS",
    "SEATS",
    "DEPARTURES_SCHEDULED",
    "DEPARTURES_PERFORMED"
]
]

tpa_mco_2025_03

,UNIQUE_CARRIER,UNIQUE_CARRIER_NAME,CARRIER,CARRIER_NAME,UNIQUE_CARRIER_ENTITY,REGION,AIRCRAFT_TYPE,AIRCRAFT_CONFIG,DATA_SOURCE,DISTANCE,PASSENGERS,SEATS,DEPARTURES_SCHEDULED,DEPARTURES_PERFORMED
599578,WN,Southwest Airlines Co.,WN,Southwest Airlines Co.,06725,D,838,1,DU,81.0,77.0,175.0,0.0,1.0
599946,WN,Southwest Airlines Co.,WN,Southwest Airlines Co.,11033,L,838,1,DU,81.0,156.0,175.0,0.0,1.0


In [18]:
mco_tpa_2025_06 = wn_f_historical.loc[
    (wn_f_historical["YEAR"] == 2025) &
    (wn_f_historical["MONTH"] == 6) &
    (wn_f_historical["ORIGIN"] == "MCO") &
    (wn_f_historical["DEST"] == "TPA"),
    [
        "UNIQUE_CARRIER_ENTITY",
        "REGION",
        "AIRCRAFT_GROUP",
        "AIRCRAFT_TYPE",
        "AIRCRAFT_CONFIG",
        "DATA_SOURCE",
        "DISTANCE",
        "PASSENGERS",
        "SEATS",
        "DEPARTURES_SCHEDULED",
        "DEPARTURES_PERFORMED"
    ]
]

comparison = mco_tpa_2025_06.reset_index(drop=True).T

different_fields = comparison[
    comparison.nunique(axis=1, dropna=False) > 1
]

different_fields.columns = ["ROW_1", "ROW_2"]

different_fields

,ROW_1,ROW_2
UNIQUE_CARRIER_ENTITY,11033,06725
REGION,L,D
PASSENGERS,25.0,284.0
SEATS,175.0,350.0
DEPARTURES_PERFORMED,1.0,2.0


In [19]:
entity_nunique = (
    wn_f_historical
    .groupby(["YEAR", "MONTH", "ORIGIN", "DEST"])["UNIQUE_CARRIER_ENTITY"]
    .nunique()
    .reset_index(name="ENTITY_NUNIQUE")
)
entity_nunique.head(10)

,YEAR,MONTH,ORIGIN,DEST,ENTITY_NUNIQUE
0,2024,1,ABQ,AUS,1
1,2024,1,ABQ,BUR,1
2,2024,1,ABQ,BWI,1
3,2024,1,ABQ,DAL,1
4,2024,1,ABQ,DEN,1
5,2024,1,ABQ,HOU,1
6,2024,1,ABQ,LAS,1
7,2024,1,ABQ,LAX,1
8,2024,1,ABQ,LGB,1
9,2024,1,ABQ,MCI,1


In [20]:
entity_nunique["ENTITY_NUNIQUE"].value_counts().sort_index()

ENTITY_NUNIQUE
1    55205
2      190
Name: count, dtype: int64

In [21]:
route_month_entity_summary = route_month_row_counts.merge(
    entity_nunique,
    how="inner",
    on=["YEAR", "MONTH", "ORIGIN", "DEST"]
)
multi_entity_route_months = route_month_entity_summary[
    route_month_entity_summary["ENTITY_NUNIQUE"] > 1
]
multi_entity_route_months["RAW_ROW_COUNT"].value_counts().sort_index()

RAW_ROW_COUNT
2      7
3     13
4    144
5     23
6      3
Name: count, dtype: int64

In [22]:
route_month_dimension_summary = (
    route_month_row_counts
    .merge(
        aircraft_type_nunique,
        on=["YEAR", "MONTH", "ORIGIN", "DEST"],
        how="inner"
    )
    .merge(
        entity_nunique,
        on=["YEAR", "MONTH", "ORIGIN", "DEST"],
        how="inner"
    )
)
route_month_dimension_summary = route_month_dimension_summary.rename(columns={
    "AIRCRAFT_TYPE_NUNIQUE_x": "AIRCRAFT_TYPE_NUNIQUE"})
route_month_dimension_summary = route_month_dimension_summary.drop(columns=["AIRCRAFT_TYPE_NUNIQUE_y"])

In [23]:
two_row_route_months = route_month_dimension_summary[
    route_month_dimension_summary["RAW_ROW_COUNT"] == 2
]
two_row_route_months

,YEAR,MONTH,ORIGIN,DEST,RAW_ROW_COUNT,AIRCRAFT_TYPE_NUNIQUE,ENTITY_NUNIQUE
10,2024,1,ABQ,MCO,2,2,1
24,2024,1,AMA,AUS,2,2,1
44,2024,1,ATL,IAD,2,2,1
83,2024,1,AUS,CUN,2,2,1
121,2024,1,BDL,DCA,2,2,1
...,...,...,...,...,...,...,...
55363,2026,5,TUL,AUS,2,2,1
55387,2026,5,TYS,MCO,2,2,1
55389,2026,5,VPS,BWI,2,2,1
55393,2026,5,VPS,MCI,2,2,1


In [24]:
two_row_route_months.groupby(
    ["AIRCRAFT_TYPE_NUNIQUE", "ENTITY_NUNIQUE"]
).size()

AIRCRAFT_TYPE_NUNIQUE  ENTITY_NUNIQUE
1                      2                    2
2                      1                 6457
                       2                    5
dtype: int64

In [25]:
mixed_two_row = two_row_route_months[
    (two_row_route_months["AIRCRAFT_TYPE_NUNIQUE"] == 2) &
    (two_row_route_months["ENTITY_NUNIQUE"] == 2)
]

mixed_two_row

,YEAR,MONTH,ORIGIN,DEST,RAW_ROW_COUNT,AIRCRAFT_TYPE_NUNIQUE,ENTITY_NUNIQUE
14968,2024,8,PBI,MCO,2,2,2
15181,2024,8,RSW,MCO,2,2,2
15224,2024,8,SAT,AUS,2,2,2
30874,2025,5,IAD,BWI,2,2,2
39143,2025,9,JAX,MCO,2,2,2


In [26]:
pbi_mco_2024_08 = wn_f_historical.loc[
    (wn_f_historical["YEAR"] == 2024) &
    (wn_f_historical["MONTH"] == 8) &
    (wn_f_historical["ORIGIN"] == "PBI") &
    (wn_f_historical["DEST"] == "MCO")
 
]

comparison = pbi_mco_2024_08.reset_index(drop=True).T

different_fields = comparison[
    comparison.nunique(axis=1, dropna=False) > 1
]

different_fields.columns = ["ROW_1", "ROW_2"]

different_fields

,ROW_1,ROW_2
PAYLOAD,34600.0,43400.0
SEATS,143.0,175.0
PASSENGERS,137.0,173.0
RAMP_TO_RAMP,54.0,56.0
AIR_TIME,31.0,38.0
UNIQUE_CARRIER_ENTITY,06725,11033
REGION,D,L
AIRCRAFT_TYPE,612,838


In [27]:
three_row_route_months = route_month_dimension_summary[
    route_month_dimension_summary["RAW_ROW_COUNT"] == 3
]

three_row_route_months.groupby(
    ["AIRCRAFT_TYPE_NUNIQUE", "ENTITY_NUNIQUE"]
).size()

AIRCRAFT_TYPE_NUNIQUE  ENTITY_NUNIQUE
2                      2                     8
3                      1                 38329
                       2                     5
dtype: int64

In [28]:
example_3row_2type_2entity = three_row_route_months[
    (three_row_route_months["AIRCRAFT_TYPE_NUNIQUE"] == 2) &
    (three_row_route_months["ENTITY_NUNIQUE"] == 2)
].iloc[0]

example_3row_2type_2entity

YEAR                     2024
MONTH                       6
ORIGIN                    PBI
DEST                      MCO
RAW_ROW_COUNT               3
AIRCRAFT_TYPE_NUNIQUE       2
ENTITY_NUNIQUE              2
Name: 10850, dtype: object

In [29]:
pbi_mco_2024_06 = wn_f_historical.loc[
    (wn_f_historical["YEAR"] == 2024) &
    (wn_f_historical["MONTH"] == 6) &
    (wn_f_historical["ORIGIN"] == "PBI") &
    (wn_f_historical["DEST"] == "MCO")
 
]

comparison = pbi_mco_2024_06.reset_index(drop=True).T

different_fields = comparison[
    comparison.nunique(axis=1, dropna=False) > 1
]

different_fields.columns = ["ROW_1", "ROW_2", "ROW_3"]

different_fields

,ROW_1,ROW_2,ROW_3
DEPARTURES_PERFORMED,1.0,1.0,2.0
PAYLOAD,34600.0,43400.0,86800.0
SEATS,143.0,175.0,350.0
PASSENGERS,143.0,175.0,347.0
RAMP_TO_RAMP,55.0,67.0,178.0
AIR_TIME,36.0,31.0,63.0
UNIQUE_CARRIER_ENTITY,06725,11033,06725
REGION,D,L,D
AIRCRAFT_TYPE,612,838,838


In [30]:
example_3row_3type_2entity = three_row_route_months[
    (three_row_route_months["AIRCRAFT_TYPE_NUNIQUE"] == 3) &
    (three_row_route_months["ENTITY_NUNIQUE"] == 2)
].iloc[0]

example_3row_3type_2entity

YEAR                     2024
MONTH                       6
ORIGIN                    AMA
DEST                      DEN
RAW_ROW_COUNT               3
AIRCRAFT_TYPE_NUNIQUE       3
ENTITY_NUNIQUE              2
Name: 9399, dtype: object

In [31]:
ama_den_2024_06 = wn_f_historical.loc[
    (wn_f_historical["YEAR"] == 2024) &
    (wn_f_historical["MONTH"] == 6) &
    (wn_f_historical["ORIGIN"] == "AMA") &
    (wn_f_historical["DEST"] == "DEN")
 
]

comparison = ama_den_2024_06.reset_index(drop=True).T

different_fields = comparison[
    comparison.nunique(axis=1, dropna=False) > 1
]

different_fields.columns = ["ROW_1", "ROW_2", "ROW_3"]

different_fields

,ROW_1,ROW_2,ROW_3
DEPARTURES_SCHEDULED,0.0,0.0,1.0
DEPARTURES_PERFORMED,1.0,1.0,2.0
PAYLOAD,43400.0,43400.0,69200.0
SEATS,175.0,175.0,286.0
PASSENGERS,158.0,174.0,214.0
FREIGHT,0.0,159.0,0.0
RAMP_TO_RAMP,76.0,65.0,133.0
AIR_TIME,55.0,56.0,110.0
UNIQUE_CARRIER_ENTITY,11033,06725,06725
REGION,L,D,D


In [32]:
four_row_route_months = route_month_dimension_summary[
    route_month_dimension_summary["RAW_ROW_COUNT"] == 4
]

four_row_route_months.groupby(
    ["AIRCRAFT_TYPE_NUNIQUE", "ENTITY_NUNIQUE"]
).size()

AIRCRAFT_TYPE_NUNIQUE  ENTITY_NUNIQUE
2                      2                   1
3                      2                 143
dtype: int64

In [33]:
example_4row_2type_2entity = four_row_route_months[
    (four_row_route_months["AIRCRAFT_TYPE_NUNIQUE"] == 2) &
    (four_row_route_months["ENTITY_NUNIQUE"] == 2)
].iloc[0]

example_4row_2type_2entity

YEAR                     2025
MONTH                       8
ORIGIN                    TPA
DEST                      MCO
RAW_ROW_COUNT               4
AIRCRAFT_TYPE_NUNIQUE       2
ENTITY_NUNIQUE              2
Name: 38333, dtype: object

In [34]:
tpa_mco_2025_08 = wn_f_historical.loc[
    (wn_f_historical["YEAR"] == 2025) &
    (wn_f_historical["MONTH"] == 8) &
    (wn_f_historical["ORIGIN"] == "TPA") &
    (wn_f_historical["DEST"] == "MCO")
 
]

comparison = tpa_mco_2025_08.reset_index(drop=True).T

different_fields = comparison[
    comparison.nunique(axis=1, dropna=False) > 1
]

different_fields.columns = ["ROW_1", "ROW_2", "ROW_3", "ROW_4"]

different_fields

,ROW_1,ROW_2,ROW_3,ROW_4
DEPARTURES_PERFORMED,1.0,1.0,3.0,5.0
PAYLOAD,34600.0,43400.0,130200.0,173000.0
SEATS,143.0,175.0,525.0,715.0
PASSENGERS,26.0,146.0,350.0,627.0
FREIGHT,0.0,0.0,664.0,1283.0
RAMP_TO_RAMP,39.0,54.0,152.0,226.0
AIR_TIME,22.0,21.0,71.0,112.0
UNIQUE_CARRIER_ENTITY,11033,11033,06725,06725
REGION,L,L,D,D
AIRCRAFT_TYPE,612,838,838,612


In [35]:
example_4row_3type_2entity = four_row_route_months[
    (four_row_route_months["AIRCRAFT_TYPE_NUNIQUE"] == 3) &
    (four_row_route_months["ENTITY_NUNIQUE"] == 2)
].iloc[0]

example_4row_3type_2entity

YEAR                     2024
MONTH                       1
ORIGIN                    BWI
DEST                      MCO
RAW_ROW_COUNT               4
AIRCRAFT_TYPE_NUNIQUE       3
ENTITY_NUNIQUE              2
Name: 273, dtype: object

In [36]:
bwi_mco_2024_01 = wn_f_historical.loc[
    (wn_f_historical["YEAR"] == 2024) &
    (wn_f_historical["MONTH"] == 1) &
    (wn_f_historical["ORIGIN"] == "BWI") &
    (wn_f_historical["DEST"] == "MCO")
 
]

comparison = bwi_mco_2024_01.reset_index(drop=True).T

different_fields = comparison[
    comparison.nunique(axis=1, dropna=False) > 1
]

different_fields.columns = ["ROW_1", "ROW_2", "ROW_3", "ROW_4"]

different_fields

,ROW_1,ROW_2,ROW_3,ROW_4
DEPARTURES_SCHEDULED,0.0,87.0,104.0,119.0
DEPARTURES_PERFORMED,1.0,84.0,103.0,114.0
PAYLOAD,34600.0,2906400.0,4470200.0,4947600.0
SEATS,143.0,12012.0,18025.0,19950.0
PASSENGERS,143.0,9392.0,13679.0,15683.0
FREIGHT,0.0,8983.0,18899.0,29860.0
RAMP_TO_RAMP,171.0,12486.0,15055.0,16617.0
AIR_TIME,142.0,10355.0,12414.0,13849.0
UNIQUE_CARRIER_ENTITY,11033,06725,06725,06725
REGION,L,D,D,D


In [37]:
five_row_route_months = route_month_dimension_summary[
    route_month_dimension_summary["RAW_ROW_COUNT"] == 5
]

five_row_route_months.groupby(
    ["AIRCRAFT_TYPE_NUNIQUE", "ENTITY_NUNIQUE"]
).size()

AIRCRAFT_TYPE_NUNIQUE  ENTITY_NUNIQUE
3                      2                 23
dtype: int64

In [38]:
example_5row_3type_2entity = five_row_route_months[
    (five_row_route_months["AIRCRAFT_TYPE_NUNIQUE"] == 3) &
    (five_row_route_months["ENTITY_NUNIQUE"] == 2)
].iloc[0]

example_5row_3type_2entity

YEAR                     2024
MONTH                       1
ORIGIN                    AUS
DEST                      HOU
RAW_ROW_COUNT               5
AIRCRAFT_TYPE_NUNIQUE       3
ENTITY_NUNIQUE              2
Name: 89, dtype: object

In [39]:
aus_hou_2024_01 = wn_f_historical.loc[
    (wn_f_historical["YEAR"] == 2024) &
    (wn_f_historical["MONTH"] == 1) &
    (wn_f_historical["ORIGIN"] == "AUS") &
    (wn_f_historical["DEST"] == "HOU")
 
]

comparison = aus_hou_2024_01.reset_index(drop=True).T

different_fields = comparison[
    comparison.nunique(axis=1, dropna=False) > 1
]

different_fields.columns = ["ROW_1", "ROW_2", "ROW_3", "ROW_4", "ROW_5"]

different_fields

,ROW_1,ROW_2,ROW_3,ROW_4,ROW_5
DEPARTURES_SCHEDULED,0.0,0.0,27.0,41.0,83.0
DEPARTURES_PERFORMED,1.0,2.0,26.0,40.0,86.0
PAYLOAD,43400.0,69200.0,1128400.0,1736000.0,2975600.0
SEATS,175.0,286.0,4550.0,7000.0,12298.0
PASSENGERS,168.0,282.0,2167.0,3292.0,6012.0
FREIGHT,0.0,0.0,6558.0,5812.0,16600.0
RAMP_TO_RAMP,125.0,144.0,1307.0,2219.0,4747.0
AIR_TIME,46.0,75.0,817.0,1281.0,2759.0
UNIQUE_CARRIER_ENTITY,11033,11033,06725,06725,06725
REGION,L,L,D,D,D


In [40]:
six_row_route_months = route_month_dimension_summary[
    route_month_dimension_summary["RAW_ROW_COUNT"] == 6
]

six_row_route_months.groupby(
    ["AIRCRAFT_TYPE_NUNIQUE", "ENTITY_NUNIQUE"]
).size()

AIRCRAFT_TYPE_NUNIQUE  ENTITY_NUNIQUE
3                      2                 3
dtype: int64

In [41]:
example_6row_3type_2entity = six_row_route_months[
    (six_row_route_months["AIRCRAFT_TYPE_NUNIQUE"] == 3) &
    (six_row_route_months["ENTITY_NUNIQUE"] == 2)
].iloc[0]

example_6row_3type_2entity

YEAR                     2025
MONTH                       5
ORIGIN                    SLC
DEST                      DEN
RAW_ROW_COUNT               6
AIRCRAFT_TYPE_NUNIQUE       3
ENTITY_NUNIQUE              2
Name: 31815, dtype: object

In [42]:
slc_den_2025_05 = wn_f_historical.loc[
    (wn_f_historical["YEAR"] == 2025) &
    (wn_f_historical["MONTH"] == 5) &
    (wn_f_historical["ORIGIN"] == "SLC") &
    (wn_f_historical["DEST"] == "DEN")
 
]

comparison = slc_den_2025_05.reset_index(drop=True).T

different_fields = comparison[
    comparison.nunique(axis=1, dropna=False) > 1
]

different_fields.columns = ["ROW_1", "ROW_2", "ROW_3", "ROW_4", "ROW_5", "ROW_6"]

different_fields

,ROW_1,ROW_2,ROW_3,ROW_4,ROW_5,ROW_6
DEPARTURES_SCHEDULED,0.0,0.0,0.0,50.0,55.0,84.0
DEPARTURES_PERFORMED,1.0,1.0,1.0,56.0,56.0,87.0
PAYLOAD,34600.0,43400.0,43400.0,2430400.0,1937600.0,3775800.0
SEATS,143.0,175.0,175.0,9800.0,8008.0,15225.0
PASSENGERS,100.0,152.0,173.0,7394.0,6706.0,12542.0
FREIGHT,0.0,0.0,0.0,24249.0,25593.0,44397.0
RAMP_TO_RAMP,86.0,76.0,89.0,4978.0,4925.0,7587.0
AIR_TIME,62.0,60.0,57.0,3563.0,3537.0,5493.0
UNIQUE_CARRIER_ENTITY,11033,11033,11033,06725,06725,06725
REGION,L,L,L,D,D,D


In [43]:
grain_field_variation = (
    wn_f_historical
    .groupby(["YEAR", "MONTH", "ORIGIN", "DEST"])
    .agg(
        AIRCRAFT_CONFIG_NUNIQUE=("AIRCRAFT_CONFIG", "nunique"),
        AIRCRAFT_GROUP_NUNIQUE=("AIRCRAFT_GROUP", "nunique"),
        DATA_SOURCE_NUNIQUE=("DATA_SOURCE", "nunique"),
        DISTANCE_NUNIQUE=("DISTANCE", "nunique")
    )
)

grain_field_variation.apply(lambda col: col.value_counts().sort_index())

,AIRCRAFT_CONFIG_NUNIQUE,AIRCRAFT_GROUP_NUNIQUE,DATA_SOURCE_NUNIQUE,DISTANCE_NUNIQUE
1,55395,55395,55395,55395


In [44]:
activity_variation = (
    wn_f_historical
    .groupby(["YEAR", "MONTH", "ORIGIN", "DEST"])
    .agg(
        RAW_ROW_COUNT=("PASSENGERS", "size"),
        PASSENGERS_NUNIQUE=("PASSENGERS", "nunique"),
        SEATS_NUNIQUE=("SEATS", "nunique"),
        DEPARTURES_SCHEDULED_NUNIQUE=("DEPARTURES_SCHEDULED", "nunique"),
        DEPARTURES_PERFORMED_NUNIQUE=("DEPARTURES_PERFORMED", "nunique")
    )
)

multi_row_activity_variation = activity_variation[
    activity_variation["RAW_ROW_COUNT"] > 1
]

(
    multi_row_activity_variation[
        [
            "PASSENGERS_NUNIQUE",
            "SEATS_NUNIQUE",
            "DEPARTURES_SCHEDULED_NUNIQUE",
            "DEPARTURES_PERFORMED_NUNIQUE"
        ]
    ] > 1
).sum()

PASSENGERS_NUNIQUE              44959
SEATS_NUNIQUE                   44455
DEPARTURES_SCHEDULED_NUNIQUE    43822
DEPARTURES_PERFORMED_NUNIQUE    43991
dtype: int64

In [45]:
wn_f_historical.duplicated().sum()

np.int64(0)

In [46]:
grain_columns = [
    "YEAR",
    "MONTH",
    "ORIGIN",
    "DEST",
    "AIRCRAFT_TYPE",
    "UNIQUE_CARRIER_ENTITY",
    "REGION",
    "AIRCRAFT_CONFIG",
    "AIRCRAFT_GROUP",
    "DATA_SOURCE",
    "DISTANCE"
]

wn_f_historical[grain_columns].isna().sum()

YEAR                     0
MONTH                    0
ORIGIN                   0
DEST                     0
AIRCRAFT_TYPE            0
UNIQUE_CARRIER_ENTITY    0
REGION                   0
AIRCRAFT_CONFIG          0
AIRCRAFT_GROUP           0
DATA_SOURCE              0
DISTANCE                 0
dtype: int64

### Phase 4.3.1 — Raw Grain Investigation Conclusions

- The working WN scheduled-service dataset contains 139,082 raw T-100 Segment rows covering 55,395 apparent directional route-months from January 2024 through May 2026.
- A raw T-100 Segment row is not equivalent to one analytical route-month. About 81% of apparent route-months contain multiple legitimate source rows.
- Route-month multiplicity is primarily explained by `AIRCRAFT_TYPE` and, in a smaller number of cases, `UNIQUE_CARRIER_ENTITY` / `REGION`.
- Different entity × aircraft-type combinations appear only when corresponding activity is reported; not every theoretical combination must exist.
- `AIRCRAFT_CONFIG`, `AIRCRAFT_GROUP`, `DATA_SOURCE`, and `DISTANCE` do not vary within the apparent route-month grain and therefore do not explain multiplicity.
- `PASSENGERS`, `SEATS`, `DEPARTURES_SCHEDULED`, and `DEPARTURES_PERFORMED` usually differ across legitimate raw rows within multi-row route-months, indicating that the rows represent separate portions of reported activity rather than duplicate route-month totals.
- No exact duplicate raw rows were found.
- No null values were found in the route-month key or grain-explanation fields inspected.
- Some inspected raw records contain performed departures even when scheduled departures are zero. These records are retained unchanged during Phase 4.3.1.
- No final route-month aggregation rules are defined in this phase.